# Recovery Blueprint M5: Kaggle setup and GPU preflight

**Use this notebook first.** It chooses the reviewed package versions, checks
your actual GPU image, runs the preparation tests, and saves one results ZIP.
Prepared 8 October 2026.

1. Import this file with **File → Import Notebook** in Kaggle.
2. Keep the notebook private. In **Settings**, select **GPU T4 x2** and enable **Internet**.
3. Upload **m5-kaggle-preflight.zip** as a **private dataset** and attach it with **Add Input**.
   Use the ZIP supplied alongside this notebook; its name and contents can stay as provided.
4. Choose **Run All**. You do not need to edit cells or choose versions.
5. Download **m5-kaggle-preflight-results.zip** from the final cell, stop the
   Kaggle session, and send back the ZIP or the final summary.

A missing input or failed setup still produces a report. The notebook completes
setup and host preparation before weight downloads or model generation.
T4 model loading, worst-context VRAM/headroom, finite model outputs and the
bounded 24-attempt smoke remain the following gate.

**Expected starting model:** Qwen/Qwen3.5-9B, pinned revision.
The reviewed 4B fallback is shown for storage planning; model selection remains
governed by the existing hardware/capability gates.

## 1. Start a recoverable preflight

Each Run All uses a new output directory. Existing files and Kaggle's base
Python environment are preserved. The notebook uses subprocesses for package
checks, so there is no kernel restart or notebook-local model factory.

Record the provider's GPU assignment and release times when available.
Notebook execution time is a lower bound on allocation; it cannot prove when
Kaggle started or released the GPUs. The existing smoke/development resource
limits still apply.

In [1]:
import csv, io, json, os, platform, shutil, sqlite3, stat, subprocess, sys, time, zipfile
from datetime import datetime, timezone
from contextlib import closing
from importlib.metadata import version, PackageNotFoundError
from pathlib import Path, PurePosixPath
from urllib.request import urlopen

EXPECTED_COMMIT = "ca46bcd55486b38bb453f5294e01f943bc7f7752"
WORK_ROOT = Path(os.environ.get("RECOVERY_PREFLIGHT_WORKDIR", "/kaggle/working"))
INPUT_ROOT = Path(os.environ.get("RECOVERY_PREFLIGHT_INPUTDIR", "/kaggle/input"))
WORK_ROOT.mkdir(parents=True, exist_ok=True)
RUN_ROOT = WORK_ROOT / ("recovery-m5-preflight-" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ"))
RUN_ROOT.mkdir(exist_ok=False)
SOURCE_ROOT = RUN_ROOT / "source"
ENV_ROOT = RUN_ROOT / "candidate-env"
LOG_ROOT = RUN_ROOT / "logs"
LOG_ROOT.mkdir()
STARTED = time.monotonic()
DEADLINE = STARTED + 780  # Leave the packet's 120-second shutdown allowance.
PINNED = {
    "pydantic": "2.13.5", "langgraph": "1.2.12",
    "langgraph-checkpoint": "4.2.0", "langgraph-checkpoint-sqlite": "3.1.1",
    "langchain-core": "1.6.6", "transformers": "5.12.0",
    "accelerate": "1.14.0", "bitsandbytes": "0.49.2",
    "tokenizers": "0.22.2", "huggingface-hub": "1.19.0",
}
MODEL_REVISIONS = {
    "9B": ("Qwen/Qwen3.5-9B", "c202236235762e1c871ad0ccb60c8ee5ba337b9a"),
    "4B": ("Qwen/Qwen3.5-4B", "851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a"),
}
report = {
    "scope": "M5 Kaggle setup/preflight; model smoke unexecuted",
    "inspection_utc": datetime.now(timezone.utc).isoformat(),
    "source_commit_expected": EXPECTED_COMMIT,
    "checks": [], "notes": [], "states": {},
    "allocation": {"provider_assignment_utc": None, "provider_release_utc": None,
                   "observed_seconds_are_lower_bound": True},
    "scientific_gates": {"model_loading": None, "model_finite_outputs": None,
                         "worst_context_headroom": None, "smoke": None,
                         "capability_pilot": None, "main_execution": None},
}

def save_json(name, value):
    (RUN_ROOT / name).write_text(json.dumps(value, indent=2, ensure_ascii=True) + "\n", encoding="utf-8")

def command(label, args, *, cwd=None, env=None, timeout=120):
    remaining = DEADLINE - time.monotonic()
    entry = {"name": label, "argv": [str(x) for x in args], "passed": False}
    began = time.monotonic()
    output = stdout = ""
    try:
        if remaining <= 0:
            raise TimeoutError("The notebook workflow deadline is reached; export and stop the session.")
        result = subprocess.run([str(x) for x in args], cwd=cwd, env=env,
                                capture_output=True, text=True, encoding="utf-8",
                                errors="replace", timeout=min(timeout, remaining))
        entry["exit_code"] = result.returncode
        entry["passed"] = result.returncode == 0
        stdout = result.stdout
        output = stdout + result.stderr
    except subprocess.TimeoutExpired as error:
        entry["error"] = "bounded command timeout"
        for value in (error.stdout, error.stderr):
            if value:
                output += value.decode("utf-8", errors="replace") if isinstance(value, bytes) else value
    except (OSError, TimeoutError) as error:
        entry["error"] = str(error)
    entry["elapsed_seconds"] = round(time.monotonic() - began, 3)
    log = LOG_ROOT / (label + ".log")
    log.write_text(output, encoding="utf-8")
    entry["log"] = str(log.relative_to(RUN_ROOT))
    report["checks"].append(entry)
    save_json("preflight-report.json", report)
    print(label + ": " + ("PASS" if entry["passed"] else "FAILED — see " + entry["log"]))
    return entry["passed"], stdout

HOST_GPU_READY = SOURCE_READY = SETUP_READY = ENVIRONMENT_READY = TESTS_READY = SQLITE_READY = False
print("Output directory:", RUN_ROOT)

Output directory: /kaggle/working/recovery-m5-preflight-20261008T125104687053Z


## 2. Inspect the actual GPU image

Expect two T4 devices. This cell captures their UUIDs, driver and memory,
the base package versions and usable CUDA build. It downloads only a small
public model metadata response to check Internet access.

In [2]:
host = {"python": platform.python_version(), "os": platform.platform(),
        "sqlite": sqlite3.sqlite_version, "packages": {}, "filesystems": {},
        "cuda_visible_devices": os.environ.get("CUDA_VISIBLE_DEVICES")}
for name in ["torch", *PINNED]:
    try:
        host["packages"][name] = version(name)
    except PackageNotFoundError:
        host["packages"][name] = "not installed"
for path in [Path("/"), Path("/tmp"), WORK_ROOT, INPUT_ROOT]:
    if path.exists():
        host["filesystems"][str(path)] = {
            "device": path.stat().st_dev,
            "free_gib": round(shutil.disk_usage(path).free / 2**30, 2),
        }
ok, output = command("host-nvidia-smi", [
    "nvidia-smi", "--query-gpu=index,uuid,name,driver_version,memory.total,memory.free",
    "--format=csv,noheader,nounits",
], timeout=30)
gpu_rows = []
if ok:
    try:
        for row in csv.reader(io.StringIO(output)):
            if row:
                if len(row) != 6:
                    raise ValueError("Unexpected nvidia-smi row.")
                gpu_rows.append(dict(zip(["index", "uuid", "name", "driver", "memory_mib", "free_mib"],
                                         (x.strip() for x in row))))
        HOST_GPU_READY = len(gpu_rows) == 2 and all("T4" in row["name"] for row in gpu_rows)
    except ValueError as error:
        report["notes"].append(str(error))
host["gpus"] = gpu_rows
probe = (
    "import json,torch; "
    "print(json.dumps(dict(version=torch.__version__,cuda_build=torch.version.cuda,"
    "cuda_available=torch.cuda.is_available(),device_count=torch.cuda.device_count(),"
    "compiled_architectures=torch.cuda.get_arch_list())))"
)
ok, output = command("host-torch", [sys.executable, "-c", probe], timeout=45)
host["torch_runtime"] = {}
if ok:
    try:
        host["torch_runtime"] = json.loads(output)
    except json.JSONDecodeError:
        report["notes"].append("Base Torch inspection did not return valid JSON.")
internet_ready = False
try:
    model, revision = MODEL_REVISIONS["9B"]
    with urlopen(f"https://huggingface.co/api/models/{model}/revision/{revision}", timeout=15) as response:
        internet_ready = response.status == 200
        host["internet_http_status"] = response.status
except Exception as error:
    host["internet_error"] = f"{type(error).__name__}: {error}"
host["internet_ready"] = internet_ready
report["host"] = host
report["states"]["two_t4_devices"] = HOST_GPU_READY
save_json("host.json", host)
save_json("preflight-report.json", report)
print(json.dumps(host, indent=2))
if not HOST_GPU_READY:
    print("NEXT ACTION: select GPU T4 x2 in Kaggle Settings, then run the notebook again.")
if not internet_ready:
    print("NEXT ACTION: enable Internet in Kaggle Settings, then run the notebook again.")

host-nvidia-smi: PASS
host-torch: PASS
{
  "python": "3.13.15",
  "os": "Linux-6.18.48+-x86_64-with-glibc2.39",
  "sqlite": "3.45.1",
  "packages": {
    "torch": "2.11.0+cu128",
    "pydantic": "2.13.5",
    "langgraph": "1.2.11",
    "langgraph-checkpoint": "4.2.0",
    "langgraph-checkpoint-sqlite": "not installed",
    "langchain-core": "1.6.1",
    "transformers": "5.16.1",
    "accelerate": "1.14.0",
    "bitsandbytes": "not installed",
    "tokenizers": "0.23.1",
    "huggingface-hub": "1.29.0"
  },
  "filesystems": {
    "/": {
      "device": 54,
      "free_gib": 1069.91
    },
    "/tmp": {
      "device": 54,
      "free_gib": 1069.91
    },
    "/kaggle/working": {
      "device": 1794,
      "free_gib": 19.5
    },
    "/kaggle/input": {
      "device": 1794,
      "free_gib": 19.5
    }
  },
  "cuda_visible_devices": null,
  "gpus": [
    {
      "index": "0",
      "uuid": "GPU-62709afd-94a6-5925-d476-6f89b2d91d8c",
      "name": "Tesla T4",
      "driver": "580.178.04"

## 3. Find and unpack the attached source bundle

The notebook handles both a ZIP input and Kaggle's automatically extracted
dataset layout. The supplied source revision is checked before running code.
If the input is missing, attach the companion ZIP as a private dataset and
run again; the host report will still be downloadable below.

In [3]:
def safe_archive(bundle, destination):
    for item in bundle.infolist():
        path = PurePosixPath(item.filename)
        if path.is_absolute() or ".." in path.parts or "\\" in item.filename or ":" in item.filename:
            raise ValueError("Unsafe archive path: " + item.filename)
        if stat.S_ISLNK(item.external_attr >> 16):
            raise ValueError("Archive symlinks are not accepted.")
        target = (destination / item.filename).resolve()
        if not target.is_relative_to(destination.resolve()):
            raise ValueError("Archive path leaves the source directory.")
    bundle.extractall(destination)

try:
    archives = sorted(INPUT_ROOT.rglob("m5-kaggle-preflight.zip")) if INPUT_ROOT.exists() else []
    folders = sorted({p.parent for p in INPUT_ROOT.rglob("SOURCE_COMMIT.txt")}) if INPUT_ROOT.exists() else []
    if len(archives) == 1:
        with zipfile.ZipFile(archives[0]) as bundle:
            if bundle.read("SOURCE_COMMIT.txt").decode("utf-8").strip() != EXPECTED_COMMIT:
                raise ValueError("The attached source revision differs from the supplied bundle.")
            SOURCE_ROOT.mkdir(exist_ok=False)
            safe_archive(bundle, SOURCE_ROOT)
        report["source_input"] = str(archives[0])
    elif not archives and len(folders) == 1:
        folder = folders[0]
        if (folder / "SOURCE_COMMIT.txt").read_text(encoding="utf-8").strip() != EXPECTED_COMMIT:
            raise ValueError("The extracted source revision differs from the supplied bundle.")
        if any(path.is_symlink() for path in folder.rglob("*")):
            raise ValueError("Input source symlinks are not accepted.")
        shutil.copytree(folder, SOURCE_ROOT)
        report["source_input"] = str(folder)
    elif not archives and not folders:
        raise FileNotFoundError("Attach m5-kaggle-preflight.zip using Kaggle Add Input.")
    else:
        raise ValueError("Multiple source bundles found. Attach only the supplied dataset.")
    required = ["pyproject.toml", "recovery/inference.py", "tests/fixtures/population_target.json",
                "tests/fixtures/schedules_target.json", "tests/fixtures/capacity_report.json.gz"]
    required += ["metadata/" + model + "/" + name for model in ("9B", "4B")
                 for name in ("config.json", "tokenizer_config.json", "tokenizer.json",
                              "chat_template.jinja", "model.safetensors.index.json")]
    missing = [name for name in required if not (SOURCE_ROOT / name).is_file()]
    if missing:
        raise ValueError("The input bundle is incomplete: " + ", ".join(missing))
    report["source_commit"] = (SOURCE_ROOT / "SOURCE_COMMIT.txt").read_text().strip()
    SOURCE_READY = True
    print("Source bundle ready.")
except Exception as error:
    report["notes"].append(f"Source input: {type(error).__name__}: {error}")
    print("Source input needs attention:", error)
report["states"]["source_ready"] = SOURCE_READY
save_json("preflight-report.json", report)

Source input needs attention: The input bundle is incomplete: tests/fixtures/capacity_report.json.gz


## 4. Install the reviewed candidate into a separate environment

The notebook fixes the versions for you. It reuses a working Torch 2.11 CUDA
base when suitable; otherwise it installs the official CUDA 12.6 build in a
clean environment. The base notebook packages are preserved.

A clean CUDA installation can take several minutes and use several GiB.
A driver, installation, package-check or time-limit failure is saved in the
results; dependent checks will be skipped.

In [4]:
candidate_env = os.environ.copy()
candidate_env.update(PYTHONPATH=str(SOURCE_ROOT), PYTHONDONTWRITEBYTECODE="1",
                     TOKENIZERS_PARALLELISM="false", PYTHONUTF8="1",
                     HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1")
ENV_PYTHON = ENV_ROOT / ("Scripts/python.exe" if os.name == "nt" else "bin/python")
runtime = host.get("torch_runtime", {})
base_version = str(runtime.get("version", "")).split("+")[0]
reuse_torch = (
    base_version == "2.11.0" and runtime.get("cuda_available") is True
    and runtime.get("cuda_build") in ("12.6", "12.8", "13.0")
    and "sm_75" in runtime.get("compiled_architectures", [])
    and runtime.get("device_count") == 2
)
driver_ok = False
try:
    driver_ok = bool(gpu_rows) and all(
        tuple(int(x) for x in row["driver"].split(".")[:3]) >= (560, 28, 3)
        for row in gpu_rows
    )
except ValueError:
    pass
if SOURCE_READY and HOST_GPU_READY and internet_ready and sys.version_info >= (3, 11):
    if reuse_torch or driver_ok:
        create_args = [sys.executable, "-m", "venv"]
        if reuse_torch:
            create_args.append("--system-site-packages")
        create_args.append(ENV_ROOT)
        ok, _ = command("create-candidate-env", create_args, timeout=45)
        if ok and not reuse_torch:
            ok, _ = command("install-cuda-torch", [
                ENV_PYTHON, "-m", "pip", "install", "--no-cache-dir",
                "--disable-pip-version-check", "torch==2.11.0+cu126",
                "--index-url", "https://download.pytorch.org/whl/cu126",
            ], timeout=420)
        if ok:
            requirements = [f"{name}=={value}" for name, value in PINNED.items()]
            (RUN_ROOT / "candidate-requirements.txt").write_text(
                "torch==2.11.0  # actual CUDA build is recorded separately\n" + "\n".join(requirements) + "\n",
                encoding="utf-8")
            ok, _ = command("install-candidate-packages", [
                ENV_PYTHON, "-m", "pip", "install", "--no-cache-dir",
                "--disable-pip-version-check", *requirements,
            ], timeout=240)
        if ok:
            ok, _ = command("install-source", [
                ENV_PYTHON, "-m", "pip", "install", "--no-deps", "--no-cache-dir",
                "--disable-pip-version-check", SOURCE_ROOT,
            ], timeout=60)
        SETUP_READY = ok
        report["environment_mode"] = "reuse-compatible-base-torch" if reuse_torch else "clean-cuda126"
    else:
        report["notes"].append("The driver does not meet the conservative CUDA 12.6 installation prerequisite.")
        print("Driver prerequisite failed. Download the report for review.")
else:
    print("Setup skipped: the GPU, Internet, Python or source input prerequisite is missing.")
report["states"]["package_install_completed"] = SETUP_READY
save_json("preflight-report.json", report)

Setup skipped: the GPU, Internet, Python or source input prerequisite is missing.


## 5. Verify versions, CUDA and the maintained text loader

This imports the maintained model class without constructing a model.
The CUDA check uses one tiny tensor per GPU. Passing it verifies basic CUDA
operation; finite LLM scores and model VRAM headroom remain unmeasured.

In [5]:
if SETUP_READY:
    check_script = RUN_ROOT / "check_environment.py"
    check_script.write_text(
        "import json, os, sys\n"
        "from importlib.metadata import version\n"
        "from pathlib import Path\n"
        "import torch\n"
        "from transformers.models.qwen3_5.modeling_qwen3_5 import Qwen3_5ForCausalLM\n"
        f"expected = {PINNED!r}\n"
        "actual = {name: version(name) for name in expected}\n"
        "assert actual == expected, (actual, expected)\n"
        "assert version('torch').split('+')[0] == '2.11.0'\n"
        "assert torch.cuda.is_available() and torch.cuda.device_count() == 2\n"
        "devices = []\n"
        "for i in range(2):\n"
        "    props = torch.cuda.get_device_properties(i)\n"
        "    assert 'T4' in props.name and (props.major, props.minor) == (7, 5)\n"
        "    value = torch.ones(1, device=f'cuda:{i}') + 1\n"
        "    assert torch.isfinite(value).all().item() and value.item() == 2\n"
        "    torch.cuda.synchronize(i)\n"
        "    devices.append(dict(index=i, name=props.name, uuid=str(getattr(props,'uuid',None)),\n"
        "                        total_bytes=props.total_memory, capability=[props.major,props.minor]))\n"
        "    del value\n"
        "    torch.cuda.empty_cache()\n"
        "data = dict(python=sys.version, packages=actual, torch=version('torch'),\n"
        "            cuda_build=torch.version.cuda, devices=devices,\n"
        "            compiled_architectures=torch.cuda.get_arch_list(),\n"
        "            text_class=Qwen3_5ForCausalLM.__name__, model_constructed=False)\n"
        "assert data['text_class'] == 'Qwen3_5ForCausalLM'\n"
        "print(json.dumps(data, indent=2))\n"
        f"Path({str(RUN_ROOT / 'candidate-environment.json')!r}).write_text(json.dumps(data,indent=2)+'\\n')\n",
        encoding="utf-8")
    imports_ok, _ = command("candidate-cuda-and-imports", [ENV_PYTHON, check_script],
                            cwd=SOURCE_ROOT, env=candidate_env, timeout=90)
    packages_ok, _ = command("candidate-pip-check", [ENV_PYTHON, "-m", "pip", "check"],
                             env=candidate_env, timeout=30)
    command("candidate-package-list", [ENV_PYTHON, "-m", "pip", "list", "--format=json",
                                      "--disable-pip-version-check"], env=candidate_env, timeout=30)
    ENVIRONMENT_READY = imports_ok and packages_ok
else:
    print("Environment verification skipped because installation did not complete.")
report["states"]["candidate_environment_verified"] = ENVIRONMENT_READY
save_json("preflight-report.json", report)

Environment verification skipped because installation did not complete.


## 6. Run the protocol, persistence and native tokenizer checks

The first command contains 81 focused deterministic tests of the adapter,
replacement/checkpoints, bounded process recovery and worker/backup behavior.
The next commands add four native checks for each model and eight capacity
checks. They construct no language model and sample no replies.

The capacity test deliberately checks an overflowing input; that warning is
expected when its guard test passes. Other failures are retained in the logs.

In [6]:
if ENVIRONMENT_READY:
    focused_ok, _ = command("focused-81", [
        ENV_PYTHON, "-m", "unittest", "tests.test_inference_preparation",
        "tests.inference_checks", "tests.durable_checks",
        "tests.attempt_checks", "tests.worker_checks", "-q",
    ], cwd=SOURCE_ROOT, env=candidate_env, timeout=360)
    native_results = []
    if focused_ok:
        for model in ("9B", "4B"):
            passed, _ = command("native-" + model, [
                ENV_PYTHON, "-m", "tests.tokenizer_inference", "--tokenizer", "metadata/" + model,
            ], cwd=SOURCE_ROOT, env=candidate_env, timeout=90)
            native_results.append(passed)
            if not passed:
                break
        if len(native_results) == 2 and all(native_results):
            capacity_ok, _ = command("native-capacity", [
                ENV_PYTHON, "-m", "tests.tokenizer_capacity", "--tokenizer", "metadata/9B",
            ], cwd=SOURCE_ROOT, env=candidate_env, timeout=90)
            TESTS_READY = capacity_ok
    print("Preparation tests:", "all passed" if TESTS_READY else "incomplete or failed; review the retained logs")
else:
    print("Preparation tests skipped because candidate environment verification did not pass.")
report["states"]["preparation_tests_verified"] = TESTS_READY
save_json("preflight-report.json", report)

Preparation tests skipped because candidate environment verification did not pass.


## 7. Verify local SQLite storage and backup

This uses the actual writable output directory with WAL and FULL
synchronization, creates a backup through the SQLite API and reopens it
read-only. These small diagnostic files are kept in the results ZIP.

In [7]:
database = RUN_ROOT / "filesystem-check.sqlite"
backup = RUN_ROOT / "filesystem-check-backup.sqlite"
try:
    with database.open("xb"):
        pass
    with closing(sqlite3.connect(database)) as connection:
        journal = connection.execute("PRAGMA journal_mode=WAL").fetchone()[0]
        connection.execute("PRAGMA synchronous=FULL")
        synchronous = connection.execute("PRAGMA synchronous").fetchone()[0]
        assert journal.lower() == "wal" and synchronous == 2
        connection.execute("CREATE TABLE probe(value INTEGER NOT NULL)")
        connection.execute("INSERT INTO probe VALUES (7)")
        connection.commit()
        with backup.open("xb"):
            pass
        with closing(sqlite3.connect(backup)) as target:
            connection.backup(target)
    uri = backup.resolve().as_uri() + "?mode=ro"
    with closing(sqlite3.connect(uri, uri=True)) as reader:
        assert reader.execute("PRAGMA integrity_check").fetchone()[0] == "ok"
        assert reader.execute("SELECT value FROM probe").fetchall() == [(7,)]
        try:
            reader.execute("INSERT INTO probe VALUES (8)")
        except sqlite3.OperationalError:
            readonly_enforced = True
        else:
            raise AssertionError("Read-only backup unexpectedly accepted a write.")
    SQLITE_READY = True
    report["sqlite_storage"] = {"journal": journal, "synchronous": synchronous,
                                "backup_integrity": "ok", "readonly_enforced": readonly_enforced}
    print("SQLite WAL/FULL, backup integrity and read-only reopen: PASS")
except Exception as error:
    report["sqlite_storage"] = {"error": f"{type(error).__name__}: {error}"}
    print("SQLite filesystem check failed:", error)
report["states"]["sqlite_storage_verified"] = SQLITE_READY
save_json("preflight-report.json", report)

SQLite WAL/FULL, backup integrity and read-only reopen: PASS


## 8. Check model-download disk space

The 9B source tensors need about 17.98 GiB on disk before headers and
additional files. Four-bit GPU loading does not shrink that download.
This cell inspects writable locations after setup and includes a one-GiB
artifact allowance. No weights are downloaded.

If 9B storage is insufficient, send the report so the pinned model can be
staged as a private read-only input or another verified location prepared.
A disk problem does not automatically select the 4B model.

In [8]:
storage = {"artifact_allowance_bytes": 2**30, "locations": [], "models": {}}
for location in [WORK_ROOT, Path("/tmp")]:
    if location.is_dir():
        probe_file = location / ("recovery-write-probe-" + RUN_ROOT.name + ".tmp")
        try:
            with probe_file.open("xb") as stream:
                stream.write(b"preflight\n")
            probe_file.unlink()  # Only this newly created disposable write probe.
            storage["locations"].append({
                "path": str(location), "device": location.stat().st_dev,
                "free_bytes": shutil.disk_usage(location).free,
            })
        except OSError as error:
            storage["locations"].append({"path": str(location), "error": str(error)})
if SOURCE_READY:
    for model, (model_id, revision) in MODEL_REVISIONS.items():
        try:
            index = json.loads((SOURCE_ROOT / "metadata" / model / "model.safetensors.index.json").read_text())
            tensor_bytes = index["metadata"]["total_size"]
            minimum = tensor_bytes + storage["artifact_allowance_bytes"]
            eligible = [x["path"] for x in storage["locations"]
                        if x.get("free_bytes", -1) >= minimum]
            storage["models"][model] = {
                "model": model_id, "revision": revision, "tensor_bytes": tensor_bytes,
                "shard_count": len(set(index["weight_map"].values())),
                "tensor_plus_artifact_allowance_bytes": minimum,
                "candidate_locations": eligible,
                "exact_file_sizes_and_download": "pending; tensor estimate excludes headers/extra files",
            }
            print(f"{model}: {tensor_bytes / 2**30:.2f} GiB source tensors; "
                  f"candidate writable locations: {eligible or 'none with the artifact allowance'}")
        except (OSError, ValueError, KeyError, TypeError) as error:
            storage["models"][model] = {"error": f"{type(error).__name__}: {error}"}
            report["notes"].append(f"{model} storage estimate is unavailable: {error}")
            print(f"{model} metadata needs review; see the result report.")
else:
    print("Model disk estimates skipped because the source metadata input is missing.")
report["storage"] = storage
save_json("storage-plan.json", storage)
save_json("preflight-report.json", report)

Model disk estimates skipped because the source metadata input is missing.


## 9. Download the results and stop the session

Send back the results ZIP or the printed summary. A passing preparation
summary is evidence for the next step; the measured model smoke remains
unexecuted. Keep the ZIP outside the notebook before considering its output
durable. Saving notebook source alone does not back up these result files.

**After downloading, stop the Kaggle session and record provider release.**

In [9]:
elapsed = time.monotonic() - STARTED
count = len(host.get("gpus", []))
report["allocation"].update(
    observed_notebook_wall_seconds=round(elapsed, 3),
    observed_allocated_gpu_seconds_lower_bound=round(count * elapsed, 3) if count else None,
    provider_release="pending external confirmation",
)
required_states = ["two_t4_devices", "source_ready", "package_install_completed",
                   "candidate_environment_verified", "preparation_tests_verified",
                   "sqlite_storage_verified"]
failed = [name for name in required_states if report["states"].get(name) is not True]
summary = {
    "result": "preparation checks passed" if not failed else "preparation incomplete",
    "failed_or_unexecuted_preparation": failed,
    "observed_wall_seconds": round(elapsed, 3),
    "model_generations": 0,
    "model_smoke": "unexecuted",
    "gpu_model_headroom": "unmeasured",
    "provider_release": "record after stopping the session",
    "next_action": "download the results ZIP, stop the session, and send back the results",
}
report["summary"] = summary
save_json("summary.json", summary)
save_json("preflight-report.json", report)
results_zip = RUN_ROOT / "m5-kaggle-preflight-results.zip"
with zipfile.ZipFile(results_zip, "x", compression=zipfile.ZIP_DEFLATED) as output:
    for path in sorted(RUN_ROOT.iterdir()):
        if path.is_file() and path != results_zip:
            output.write(path, path.name)
    for path in sorted(LOG_ROOT.glob("*.log")):
        output.write(path, "logs/" + path.name)
with zipfile.ZipFile(results_zip) as exported:
    assert exported.testzip() is None
    assert "summary.json" in exported.namelist()
print(json.dumps(summary, indent=2))
print("\nDownload:", results_zip)
try:
    from IPython.display import FileLink, display
    display(FileLink(os.path.relpath(results_zip, Path.cwd())))
except ImportError:
    pass

{
  "result": "preparation incomplete",
  "failed_or_unexecuted_preparation": [
    "source_ready",
    "package_install_completed",
    "candidate_environment_verified",
    "preparation_tests_verified"
  ],
  "observed_wall_seconds": 39.774,
  "model_generations": 0,
  "model_smoke": "unexecuted",
  "gpu_model_headroom": "unmeasured",
  "provider_release": "record after stopping the session",
  "next_action": "download the results ZIP, stop the session, and send back the results"
}

Download: /kaggle/working/recovery-m5-preflight-20261008T125104687053Z/m5-kaggle-preflight-results.zip


/kaggle/working/recovery-m5-preflight-20261008T125104687053Z/m5-kaggle-preflight-results.zip

## Reference decisions

- Scientific settings and protocols: the supplied source revision's
  PROJECT_SPECIFICATION.md and the existing M5 inference packet.
- [Kaggle notebooks](https://www.kaggle.com/docs/notebooks) and
  [NVIDIA's Kaggle T4 instructions](https://docs.nvidia.com/datascience/deployment/latest/platforms/kaggle/).
- [Official PyTorch builds](https://pytorch.org/get-started/previous-versions/)
  and [CUDA 12.6 wheels](https://download.pytorch.org/whl/cu126/torch/).
- [Transformers bitsandbytes quantization](https://huggingface.co/docs/transformers/quantization/bitsandbytes).

This notebook changes no prompts, seeds, caps, mathematical sizes, replay
rules or study populations. CUDA tensor diagnostics and deterministic tests
are preparation evidence. No weight loading, model calls, capability pilot,
main experiment or accepted freeze is claimed.